In [1]:
# Lab 2: Data Cleaning & Preprocessing for YouTube Text Mining
# This code cell is the existing cell preserved with its id.
print('Lab 2 notebook loaded - see following cells for manual and code snippets')

Lab 2 notebook loaded - see following cells for manual and code snippets


In [10]:
# Helper: peek at top lines of a file to profile it
from pathlib import Path
def peek_file(filepath, n=30):
    p = Path(filepath)
    if not p.exists():
        print(f'File not found: {filepath}')
        return
    with p.open('r', encoding='utf-8', errors='replace') as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(i+1, line.rstrip())
# Example: peek_file('comments.json')
# Example: peek_file('captions.vtt')

In [11]:
import json
from pathlib import Path
import pandas as pd
def load_comments_json(filepath='comments.json'):
    p = Path(filepath)
    if not p.exists():
        print(f'No comments file found at {filepath}')
        return pd.DataFrame(columns=['raw'])
    # Try full-json load first
    try:
        with p.open('r', encoding='utf-8') as f:
            obj = json.load(f)
        # If it's a list of objects, extract text fields
        if isinstance(obj, list):
            texts = []
            for o in obj:
                if isinstance(o, dict):
                    if o.get('reply') is True:
                        continue
                    texts.append(o.get('text', str(o)))
                else:
                    texts.append(str(o))
            return pd.DataFrame({'comment_text': texts})
    except Exception:
        pass
    # Fallback: try JSON lines
    texts = []
    with p.open('r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                texts.append(line)
                continue
            if isinstance(obj, dict):
                if obj.get('reply') is True:
                    continue
                texts.append(obj.get('text', str(obj)))
            else:
                texts.append(str(obj))
    return pd.DataFrame({'comment_text': texts})
# Load now and show a small preview
comments_df = load_comments_json('comments.json')
print('Loaded', len(comments_df), 'comments')
comments_df.head().fillna('').to_dict(orient='records')

Loaded 50 comments


[{'comment_text': 'What an absolutely beautiful and emotional short documentary! 🦓✨\nBBC Earth always has a special way of capturing the soul of wildlife, but this zebra journey feels particularly touching. The way the team follows her through danger, migration, and finally forming her own family is incredibly moving.\n\nThe cinematography is stunning — every frame feels like a painting, from the dusty plains to the gentle close-ups that reveal her courage and resilience. 🌍💛\nIt’s amazing how you can portray not just survival, but also hope, connection, and the quiet strength of an animal often overlooked.\n\nThank you for telling these powerful stories with respect, beauty, and heart. Truly inspiring work! 👏📽🌿'},
 {'comment_text': 'It’s a nice wildlife video. Greetings from Botswana 😊'},
 {'comment_text': "It's a good thing the team's aircraft served a purpose for quick communication and a location isolating system."},
 {'comment_text': 'Beautiful footage — it’s incredible to see the 

In [12]:
import re
def load_captions_vtt(filepath='captions.vtt'):
    p = Path(filepath)
    if not p.exists():
        print(f'No captions file found at {filepath}')
        return pd.DataFrame(columns=['caption_sentence'])
    try:
        import webvtt
        texts = [c.text.strip() for c in webvtt.read(str(p))]
        full = ' '.join(texts)
    except Exception:
        # Manual fallback: keep only non-timestamp, non-header lines
        lines = []
        with p.open('r', encoding='utf-8', errors='replace') as f:
            for ln in f:
                ln = ln.strip()
                if not ln:
                    continue
                if '-->' in ln or ln.isdigit() or 'WEBVTT' in ln or ln.startswith('Kind:') or ln.startswith('Language:'):
                    continue
                # remove inline time tags like <00:00:04.560><c>text</c> by stripping <...> segments
                clean = re.sub(r'<[^>]+>', '', ln)
                lines.append(clean)
        full = ' '.join(lines)
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', full) if s.strip()]
    return pd.DataFrame({'caption_sentence': sentences})
captions_df = load_captions_vtt('captions.vtt')
print('Loaded', len(captions_df), 'caption sentences')
captions_df.head().to_dict(orient='records')

Loaded 39 caption sentences


[{'caption_sentence': 'The morning after the heavy rains, the The morning after the heavy rains, the\nzebra start to arrive.'},
 {'caption_sentence': 'zebra start to arrive.'},
 {'caption_sentence': 'zebra start to arrive.'},
 {'caption_sentence': "Janet's family is the first to be seen Janet's family is the first to be seen Janet's family is the first to be seen\nby the team."},
 {'caption_sentence': 'She was the first zebra ever recorded to She was the first zebra ever recorded to\nmake this migration.'}]

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# Ensure required NLTK corpora are available, download if missing
try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    print('NLTK stopwords resource missing — downloading...')
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))
try:
    lemmatizer = WordNetLemmatizer()
except Exception:
    lemmatizer = WordNetLemmatizer()
def normalize_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()
def safe_tokenize(text):
    # Try NLTK tokenizer; if missing, attempt download, then retry; finally fallback to regex
    try:
        return word_tokenize(normalize_text(text))
    except LookupError as e:
        print('NLTK tokenizer resource missing; attempting to download punkt...')
        try:
            nltk.download('punkt')
        except Exception as e2:
            print('Automatic download failed:', e2)
            return re.findall(r'\b[a-z]+\b', normalize_text(text))
        # retry after download
        try:
            return word_tokenize(normalize_text(text))
        except Exception:
            return re.findall(r'\b[a-z]+\b', normalize_text(text))
def tokenize_and_filter(text):
    tokens = safe_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return [lemmatizer.lemmatize(t) for t in tokens]
def top_n_words_from_series(series, n=20):
    words = []
    for v in series.dropna():
        words.extend(tokenize_and_filter(v))
    return Counter(words).most_common(n)
# Apply cleaning to DataFrames if they exist
if 'comments_df' in globals() and not comments_df.empty:
    comments_df['cleaned_tokens'] = comments_df['comment_text'].fillna('').apply(tokenize_and_filter)
    comments_df['length_chars'] = comments_df['comment_text'].fillna('').apply(len)
else:
    print('comments_df not available or empty')
if 'captions_df' in globals() and not captions_df.empty:
    captions_df['cleaned_tokens'] = captions_df['caption_sentence'].fillna('').apply(tokenize_and_filter)
    captions_df['length_chars'] = captions_df['caption_sentence'].fillna('').apply(len)
else:
    print('captions_df not available or empty')
print('Cleaning applied (if data present)')

In [ ]:
# Type-token ratio
def type_token_ratio(series):
    words = []
    for v in series.dropna():
        words.extend([w for w in tokenize_and_filter(v)])
    if not words:
        return 0
    return len(set(words)) / len(words)
if 'captions_df' in globals() and not captions_df.empty:
    print('Caption TTR:', type_token_ratio(captions_df['caption_sentence']))
if 'comments_df' in globals() and not comments_df.empty:
    print('Comment TTR:', type_token_ratio(comments_df['comment_text']))
# Histograms of lengths
if 'captions_df' in globals() and 'comments_df' in globals():
    plt.figure(figsize=(8,4))
    plt.hist(captions_df['length_chars'].dropna(), bins=20, alpha=0.7, label='Captions')
    plt.hist(comments_df['length_chars'].dropna(), bins=20, alpha=0.7, label='Comments')
    plt.legend()
    plt.xlabel('Length (characters)')
    plt.ylabel('Frequency')
    plt.title('Caption vs. Comment Lengths')
    plt.show()
# Top words
if 'captions_df' in globals() and not captions_df.empty:
    print('Top 20 caption words:', top_n_words_from_series(captions_df['caption_sentence'], 20))
if 'comments_df' in globals() and not comments_df.empty:
    print('Top 20 comment words:', top_n_words_from_series(comments_df['comment_text'], 20))

In [ ]:
out_comments = 'cleaned_comments.csv'
out_captions = 'cleaned_captions.csv'
if 'comments_df' in globals() and not comments_df.empty:
    comments_df.to_csv(out_comments, index=False)
    print('Saved', out_comments)
else:
    print('No comments to save')
if 'captions_df' in globals() and not captions_df.empty:
    captions_df.to_csv(out_captions, index=False)
    print('Saved', out_captions)
else:
    print('No captions to save')

## Notes & Next Steps
- If you encounter parsing errors, inspect the original files with `peek_file()` and adapt the regexes.
- For non-English data, replace the NLTK `stopwords` list with a language-appropriate list.
- If NLTK raises resource errors, re-run the NLTK download cell in the earlier notebook or this one.

In [ ]:
import pandas as pd
import json

def parse_comments_json(filepath='comments.json'):
    comments_data = []
    
    # Open the file and read line by line (NDJSON format)
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            try:
                # Parse JSON line
                obj = json.loads(line)
                
                # Extract fields based on your file's specific keys
                comments_data.append({
                    'username': obj.get('author'),
                    'timestamp_text': obj.get('time'),
                    'comment_text': obj.get('text')
                })
            except json.JSONDecodeError:
                continue
                
    return pd.DataFrame(comments_data)

# Load the data
comments_df = parse_comments_json('comments.json')
print(f"Loaded {len(comments_df)} comments.")
print(comments_df.head())